# Phase 2C - Domain-V2 Fine-Tuning (YOLOv8n Transfer Learning, Google Colab)

This notebook fine-tunes the **existing V1 eye-shield model** (`models/best.pt`,
YOLOv8n, already trained on the 240-image Phase 2B baseline) onto the newly
captured **60-image webcam target-domain** dataset (Domain V2).

It is a *fine-tuning* runbook, not a training-from-scratch runbook. The starting
weights are the V1 checkpoint `best.pt`, never the COCO seed checkpoint.

## Why this run exists

V1 produced an observed **false-safe**: a bare-eye face was predicted as
`goggles`. Class `0 goggles` means *safe*, so a false `goggles` prediction is a
safety failure, not a cosmetic one. The 60 webcam images exist to pull the model
towards the camera, lighting, distance and head-pose distribution of the real
deployment rig.

## Ground rules (do not bend these)

- The test split is evaluated **once**, at the end. It is never used for
  training, early stopping, threshold selection, augmentation selection or
  hyper-parameter tuning.
- Training reads a **test-free YAML**. The `test:` key exists only in the
  evaluation YAML, written after training is frozen.
- Nothing here writes to `models/best.pt` in the repository. V1 weights stay
  intact on your Mac; only a *new* file (`best_v2.pt`) is produced.
- **No number is reported until it is measured.** Targets in `plan.md`
  (P ≥ 0.90, R ≥ 0.85, F1 ≥ 0.87, mAP@0.5 ≥ 0.90, ≥ 15 FPS) are targets, not
  results. If a measured value falls short, the shortfall is reported.
- The V2 result is **not** an AI-01 row. `experiments/ai_metrics.csv` holds the
  locked Phase 2B result and has a frozen schema; the V2 report is a separate
  JSON with its own schema (section 14).

## Before you start

`Runtime → Change runtime type → Hardware accelerator → T4 GPU`, then
`Runtime → Run all`. The notebook aborts immediately if CUDA is missing - a CPU
training run here would be meaningless and would waste the free-tier quota.

## What you need to upload

Exactly two files, both requested by section 4:

1. `eye_shield_domain_v2.zip` - the Domain V2 dataset (already built at
   `data/processed/eye_shield_domain_v2/` and zipped to
   `/tmp/eye_shield_domain_v2.zip`).
2. `best.pt` - the **V1** checkpoint, copied from `models/best.pt`
   (6,508,202 bytes, SHA-256 `9366b73b35d4f820af401808aa4e0e58cb29a7b557991b123e5cd6a813fa3591`).

## 1. Install the pinned Ultralytics version

`ultralytics==8.4.215` is the version pinned in `requirements.txt` and the only
one verified locally. It is installed with `%pip` (not `!pip`) so the pin is
interpolated and applies to the same interpreter that runs the notebook.

In [ ]:
PINNED_ULTRALYTICS = "8.4.215"

%pip install -q "ultralytics=={PINNED_ULTRALYTICS}"

import ultralytics

print("ultralytics", ultralytics.__version__)
assert ultralytics.__version__ == PINNED_ULTRALYTICS, (
    f"version drift: {ultralytics.__version__} != {PINNED_ULTRALYTICS}"
)

## 2. Print the environment and hard-stop without a GPU

Recorded so every metric can be traced to the exact software stack and GPU. If
CUDA is unavailable this cell raises - do not continue.

In [ ]:
import json, platform, sys

import torch
import ultralytics

env = {
    "python": sys.version.split()[0],
    "ultralytics": ultralytics.__version__,
    "torch": torch.__version__,
    "torchvision": __import__("torchvision").__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_memory_gb": round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ) if torch.cuda.is_available() else None,
    "platform": platform.platform(),
}

print(json.dumps(env, indent=2))

if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU. Runtime -> Change runtime type -> T4 GPU, then Runtime -> Run all. "
        "This notebook refuses to fine-tune on CPU."
    )

print("\nOK - CUDA GPU present:", env["gpu_name"], f"({env['gpu_memory_gb']} GB)")

## 3. Upload the two required files

Details:

- `eye_shield_domain_v2.zip`: 300 images (202 train / 52 val / 46 test), balanced
  across `goggles` and `no_goggles`. Archive root is `eye_shield_domain_v2/`.
- `best.pt`: the V1 checkpoint. This is the *starting* weight file, and it is
  read-only here.

In [ ]:
from google.colab import files

print("Select BOTH files in the dialog: eye_shield_domain_v2.zip and best.pt")
uploaded = files.upload()
print("uploaded:", sorted(uploaded))

## 4. Validate the two uploads before touching them

Nothing is extracted and no training is attempted until the ZIP opens, the
archive root is what we expect, and `best.pt` is a plausible, non-empty
weights file.

In [ ]:
import hashlib
import zipfile
from pathlib import Path

DATASET_ZIP = Path("/content/eye_shield_domain_v2.zip")
V1_WEIGHTS = Path("/content/best.pt")

assert DATASET_ZIP.is_file(), f"missing upload: {DATASET_ZIP}"
assert V1_WEIGHTS.is_file(), f"missing upload: {V1_WEIGHTS}"

print(f"{DATASET_ZIP.name:<28}{DATASET_ZIP.stat().st_size:>12,} bytes")
print(f"{V1_WEIGHTS.name:<28}{V1_WEIGHTS.stat().st_size:>12,} bytes")

assert V1_WEIGHTS.stat().st_size > 1_000_000, (
    "best.pt looks truncated - re-upload the V1 checkpoint from models/best.pt"
)

v1_sha256 = hashlib.sha256(V1_WEIGHTS.read_bytes()).hexdigest()
print("\nbest.pt SHA-256:", v1_sha256)
print("expected        : 9366b73b35d4f820af401808aa4e0e58cb29a7b557991b123e5cd6a813fa3591")
if v1_sha256 != "9366b73b35d4f820af401808aa4e0e58cb29a7b557991b123e5cd6a813fa3591":
    print("WARNING: checksum differs from the recorded V1 file - confirm this is the model you mean.")

with zipfile.ZipFile(DATASET_ZIP) as zf:
    bad = zf.testzip()
    assert bad is None, f"corrupt member in ZIP: {bad}"
    names = zf.namelist()

roots = {n.split("/", 1)[0] for n in names if n.strip()}
print("\narchive roots:", sorted(roots))
assert roots == {"eye_shield_domain_v2"}, f"unexpected archive root(s): {sorted(roots)}"

print(f"archive members: {len(names)}")

data_yaml_members = [n for n in names if n.endswith("data.yaml")]
assert len(data_yaml_members) == 1, f"expected exactly one data.yaml, found {data_yaml_members}"
print("data.yaml member:", data_yaml_members[0])
print("\nOK - uploads validated")

## 5. Extract the dataset

Extracted to `/content/domain_v2/`. Any pre-existing extraction is removed first
so a stale tree can never be silently reused.

In [ ]:
import shutil

WORK = Path("/content/domain_v2")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

with zipfile.ZipFile(DATASET_ZIP) as zf:
    zf.extractall(WORK)

DATASET_ROOT = WORK / "eye_shield_domain_v2"
assert DATASET_ROOT.is_dir(), f"dataset root missing: {DATASET_ROOT}"
print("dataset root:", DATASET_ROOT)

for split in ("train", "val", "test"):
    for sub in ("images", "labels"):
        d = DATASET_ROOT / split / sub
        assert d.is_dir(), f"missing directory: {d}"
        print(f"OK  {d.relative_to(WORK)}")

## 6. Validate `data.yaml` and the class contract

The on-disk `data.yaml` shipped with the dataset carries a machine-specific
absolute `path:` (your Mac's path), which is meaningless in Colab. The class
names, however, are the source of truth and must match `config/classes.yaml`
exactly: `0 -> goggles`, `1 -> no_goggles`.

Inverting these two would silently invert every safety decision downstream, so a
mismatch aborts the run.

In [ ]:
import yaml

EXPECTED_CLASSES = {0: "goggles", 1: "no_goggles"}

SHIPPED_YAML = DATASET_ROOT / "data.yaml"
assert SHIPPED_YAML.is_file(), f"missing {SHIPPED_YAML}"
shipped = yaml.safe_load(SHIPPED_YAML.read_text())
print("shipped data.yaml as provided:")
print(SHIPPED_YAML.read_text())

raw_names = shipped.get("names")
if isinstance(raw_names, dict):
    shipped_names = {int(k): str(v) for k, v in raw_names.items()}
else:
    shipped_names = {i: str(v) for i, v in enumerate(raw_names or [])}

assert shipped_names == EXPECTED_CLASSES, (
    f"class mismatch in data.yaml: {shipped_names} != {EXPECTED_CLASSES}"
)
print("OK - data.yaml classes are exactly:", shipped_names)

## 7. Generate the Colab-safe `data.yaml`

The absolute `path:` from your Mac is replaced with the Colab path
`/content/domain_v2/eye_shield_domain_v2`. The generated file keeps `test:`,
because this YAML doubles as the evaluation descriptor.

**Training does not read this file.** Section 9 writes a separate, test-free YAML
so that the trainer has no route to `test/images` at all.

Class order is asserted after writing: if the regenerated file ever reorders the
classes, the run stops here rather than after 25 epochs.

In [ ]:
COLAB_YAML = Path("/content/domain_v2_colab.yaml")
COLAB_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": dict(EXPECTED_CLASSES),
}, sort_keys=False))

print(COLAB_YAML.read_text())

reloaded = yaml.safe_load(COLAB_YAML.read_text())
assert {int(k): str(v) for k, v in reloaded["names"].items()} == EXPECTED_CLASSES, "class order changed"
assert Path(reloaded["path"]) == DATASET_ROOT, "path not rewritten to the Colab location"
assert reloaded["test"] == "test/images", "test split missing from the evaluation YAML"

print("OK - Colab YAML written and class order preserved (0: goggles, 1: no_goggles)")

## 8. Full dataset integrity check

Every assertion below aborts the run. This is the same set of invariants
`tools/build_domain_v2_dataset.py` establishes locally, re-verified on the Colab
copy of the data.

Expected, from the local dataset: **train 202 / val 52 / test 46 = 300 images**,
balanced `{0: 107, 1: 107}` in train and roughly 26/26 and 23/25 in val/test.

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
EXPECTED_SPLITS = {"train": 202, "val": 52, "test": 46}

def images_in(split):
    d = DATASET_ROOT / split / "images"
    return sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)

def labels_in(split):
    d = DATASET_ROOT / split / "labels"
    return sorted(p for p in d.iterdir() if p.is_file() and p.suffix.lower() == ".txt")

counts = {}
class_boxes = {}
total_boxes = 0
empty_labels = 0
invalid_rows = 0
bad_class_ids = 0
bad_geometry = 0
unpaired = 0

for split, expected in EXPECTED_SPLITS.items():
    imgs = images_in(split)
    lbls = labels_in(split)
    assert len(imgs) == expected, f"{split}: expected {expected} images, found {len(imgs)}"
    assert len(lbls) == expected, f"{split}: expected {expected} labels, found {len(lbls)}"

    img_stems = {p.stem for p in imgs}
    lbl_stems = {p.stem for p in lbls}
    missing_labels = sorted(img_stems - lbl_stems)
    orphan_labels = sorted(lbl_stems - img_stems)
    unpaired += len(missing_labels) + len(orphan_labels)
    assert not missing_labels, f"{split}: images without labels: {missing_labels[:5]}"
    assert not orphan_labels, f"{split}: labels without images: {orphan_labels[:5]}"

    per_class = {0: 0, 1: 0}
    split_boxes = 0
    for lp in lbls:
        rows = [ln for ln in lp.read_text().splitlines() if ln.strip()]
        if not rows:
            empty_labels += 1
        for row in rows:
            parts = row.split()
            if len(parts) != 5:
                invalid_rows += 1
                continue
            try:
                cid = int(float(parts[0]))
                cx, cy, w, h = (float(v) for v in parts[1:])
            except ValueError:
                invalid_rows += 1
                continue
            if cid not in per_class:
                bad_class_ids += 1
                continue
            if not (0.0 <= cx <= 1.0 and 0.0 <= cy <= 1.0 and 0.0 < w <= 1.0 and 0.0 < h <= 1.0):
                bad_geometry += 1
                continue
            per_class[cid] += 1
            split_boxes += 1

    counts[split] = {"images": len(imgs), "labels": len(lbls), "boxes": split_boxes}
    class_boxes[split] = per_class
    total_boxes += split_boxes
    print(f"{split:<6}{len(imgs):>4} images{len(lbls):>5} labels{split_boxes:>6} boxes   "
          f"goggles={per_class[0]}  no_goggles={per_class[1]}")

print(f"\ntotal images: {sum(c['images'] for c in counts.values())}")
print(f"total boxes : {total_boxes}")
print(f"empty label files: {empty_labels}   invalid rows: {invalid_rows}   "
      f"bad class ids: {bad_class_ids}   bad geometry: {bad_geometry}   unpaired: {unpaired}")

assert empty_labels == 0, f"{empty_labels} empty label files"
assert invalid_rows == 0, f"{invalid_rows} malformed label rows"
assert bad_class_ids == 0, f"{bad_class_ids} rows with a class id outside {{0, 1}}"
assert bad_geometry == 0, f"{bad_geometry} rows with out-of-bounds or non-positive boxes"
assert unpaired == 0, f"{unpaired} unpaired images/labels"
for split, per_class in class_boxes.items():
    assert per_class[0] > 0 and per_class[1] > 0, f"{split} is missing one of the two classes: {per_class}"

print("\nOK - dataset integrity verified for train / val / test")

## 9. Build the test-free training YAML

The test-split firewall is structural, not a promise. `model.train()` reads splits
from whatever YAML it is given; a YAML with no `test:` key gives the trainer no
route to `test/images` while `best.pt` is being fitted.

This mirrors `training/train_yolov8n.write_train_yaml()`, which emits exactly
four keys: `path`, `train`, `val`, `names`.

In [ ]:
TRAIN_YAML = Path("/content/domain_v2_train_only.yaml")
TRAIN_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "names": dict(EXPECTED_CLASSES),
}, sort_keys=False))

TRAIN_YAML_TEXT = TRAIN_YAML.read_text()
print(TRAIN_YAML_TEXT)

assert "test" not in TRAIN_YAML_TEXT, "test split leaked into the training YAML"
assert set(yaml.safe_load(TRAIN_YAML_TEXT)) == {"path", "train", "val", "names"}, "unexpected keys"
print("OK - training YAML has no test key")

## 10. Confirm the V1 weights and their class map

The point of fine-tuning is to keep the class semantics V1 learned. If the
uploaded `best.pt` does not declare `{0: goggles, 1: no_goggles}` as a detection
model, transfer learning would corrupt the labels and the run stops here.

In [ ]:
from ultralytics import YOLO

v1_model = YOLO(str(V1_WEIGHTS))
v1_names = {int(k): str(v) for k, v in v1_model.names.items()}
print("V1 task :", v1_model.task)
print("V1 names:", v1_names)

assert v1_model.task == "detect", f"expected a detection model, got {v1_model.task}"
assert v1_names == EXPECTED_CLASSES, f"V1 class map mismatch: {v1_names} != {EXPECTED_CLASSES}"
print("OK - V1 weights are a 2-class detector with the expected class map")

## 11. Fine-tune from `best.pt`

Transfer learning settings, all fixed in advance:

| Setting | Value | Reason |
| --- | --- | --- |
| start weights | `/content/best.pt` (V1) | fine-tune the trained detector, not COCO |
| `epochs=25` | 25 | small target domain; 60 would overfit 202 images |
| `patience=8` | 8 | early stop on the val split |
| `imgsz=640` | 640 | same geometry as V1 and the deployment pipeline |
| `batch=16` | 16 | fits a 16 GB T4 at 640 px |
| `seed=42` | 42 | reproducibility |
| `deterministic=True` | True | reproducibility |
| `device=0` | 0 | the T4 |
| `workers=2` | 2 | Colab-safe dataloader count |
| `project=/content/runs`, `name=eye_shield_domain_v2` | - | artifact location |
| `exist_ok=True` | True | re-runnable |

**If CUDA runs out of memory:** set `BATCH = 8` and re-run this cell only. Reduce
`IMGSZ` only if batch 8 also fails - changing image size changes the geometry the
model was trained at and makes the comparison against V1 unfair.

Every argument is printed before training so the run is traceable from the
notebook output alone.

In [ ]:
import time

EPOCHS = 25
PATIENCE = 8
IMGSZ = 640
BATCH = 16          # Colab T4 at 640 px. Drop to 8 only if CUDA OOMs.
SEED = 42
WORKERS = 2
RUN_NAME = "eye_shield_domain_v2"

TRAIN_ARGS = {
    "data": str(TRAIN_YAML),
    "epochs": EPOCHS,
    "patience": PATIENCE,
    "imgsz": IMGSZ,
    "batch": BATCH,
    "seed": SEED,
    "deterministic": True,
    "device": 0,
    "workers": WORKERS,
    "project": "/content/runs",
    "name": RUN_NAME,
    "exist_ok": True,
    "pretrained": True,
    "optimizer": "auto",
    "cache": False,
    "plots": True,
    "save": True,
    "val": True,
    "split": "val",
    "verbose": True,
}

print("start weights:", str(V1_WEIGHTS), "(V1 checkpoint, transfer learning)")
print(json.dumps(TRAIN_ARGS, indent=2))
assert "test" not in Path(TRAIN_ARGS["data"]).read_text(), "refusing to train against a YAML with a test split"

model = YOLO(str(V1_WEIGHTS))
started = time.time()
model.train(**TRAIN_ARGS)
wall_minutes = (time.time() - started) / 60
print(f"\ntraining wall time: {wall_minutes:.1f} min")

RUN_DIR = Path(model.trainer.save_dir)
print("run dir:", RUN_DIR)

## 12. Validate the fine-tuned checkpoint

The new weights are `best_v2.pt`. V1 stays untouched at `/content/best.pt`, so
both models remain available for the A/B comparison in section 16.

In [ ]:
BEST_V2 = Path("/content/best_v2.pt")

run_best = RUN_DIR / "weights" / "best.pt"
run_last = RUN_DIR / "weights" / "last.pt"
assert run_best.is_file(), f"training did not produce {run_best}"
assert run_last.is_file(), f"training did not produce {run_last}"
shutil.copy2(run_best, BEST_V2)

print(f"{run_best.stat().st_size:>12,} bytes  run best.pt")
print(f"{BEST_V2.stat().st_size:>12,} bytes  {BEST_V2}")

v2_model = YOLO(str(BEST_V2))
v2_names = {int(k): str(v) for k, v in v2_model.names.items()}
assert v2_names == EXPECTED_CLASSES, f"fine-tuned class map changed: {v2_names}"
print("class map:", v2_names)

assert Path("/content/best.pt").stat().st_size == V1_WEIGHTS.stat().st_size, "V1 weights were modified"
print("\nOK - best_v2.pt written; V1 best.pt untouched")

print("\ntraining artifacts:")
for name in ("results.csv", "results.png", "confusion_matrix.png",
             "confusion_matrix_normalized.png", "arg.yaml", "args.yaml"):
    p = RUN_DIR / name
    print(f"  {'OK ' if p.exists() else 'MISSING':<8} {name}"
          + (f"  ({p.stat().st_size/1024:.0f} KiB)" if p.exists() else ""))

results_csv = RUN_DIR / "results.csv"
if results_csv.is_file():
    rows = results_csv.read_text().strip().splitlines()
    print("\nresults.csv columns:")
    print(rows[0])
    print(f"{len(rows) - 1} epoch rows recorded")

## 13. Evaluate `best_v2.pt` on the held-out splits

Two evaluations, both with the test-free discipline intact:

1. **Validation split** - the split early stopping used, reported for
   completeness only.
2. **Target test split** - reported **once**, with the six-image caveat in
   section 14. Nothing is tuned after this cell.

The evaluation YAML points both `val:` and `test:` at `test/images`, mirroring
`training/evaluate_model.build_test_yaml()`. The on-disk dataset is never
modified; only the YAML handed to the validator differs.

In [ ]:
VAL_YAML = Path("/content/domain_v2_val_eval.yaml")
VAL_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "val/images",
    "names": dict(EXPECTED_CLASSES),
}, sort_keys=False))

val_results = v2_model.val(
    data=str(VAL_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    plots=True,
    save_json=False,
    project="/content/runs",
    name="domain_v2_val_eval",
    exist_ok=True,
    verbose=False,
)
print("validation split evaluation complete:", val_results.box.mean_results())

In [ ]:
TEST_YAML = Path("/content/domain_v2_test_eval.yaml")
TEST_YAML.write_text(yaml.safe_dump({
    "path": str(DATASET_ROOT),
    "train": "train/images",
    "val": "test/images",     # model.val() reads whatever the YAML calls "val"
    "test": "test/images",
    "names": dict(EXPECTED_CLASSES),
}, sort_keys=False))

print(f"evaluating ONCE on {counts['test']['images']} test images "
      f"({counts['test']['images'] // 2} with webcam stills)")
test_results = v2_model.val(
    data=str(TEST_YAML),
    split="val",
    imgsz=IMGSZ,
    batch=BATCH,
    device=0,
    plots=True,
    save_json=False,
    project="/content/runs",
    name="domain_v2_test_eval",
    exist_ok=True,
    verbose=False,
)
print("target test split evaluation complete:", test_results.box.mean_results())

## 14. Extract per-class and aggregate metrics

Read from the Ultralytics `BoxMetrics` object, exactly as
`training/evaluate_model.extract_metrics()` does locally. F1 is the harmonic mean
with an explicit `P + R == 0` guard, and every value is checked finite before it
is written - a `NaN` in a report is a bug, not a result.

The test split here is **46 images**, of which only **6** are webcam stills (3
`goggles`, 3 `no_goggles`). That is enough to catch a regression, not enough to
certify field performance. The caveat is written into the report itself.

In [ ]:
import math

def extract_metrics(results, model_names):
    box = results.box
    present = list(box.ap_class_index)

    per_class = {}
    for pos, class_index in enumerate(present):
        class_index = int(class_index)
        p, r = float(box.p[pos]), float(box.r[pos])
        per_class[model_names[class_index]] = {
            "class_id": class_index,
            "precision": round(p, 6),
            "recall": round(r, 6),
            "f1": round(0.0 if p + r == 0 else 2 * p * r / (p + r), 6),
            "map50": round(float(box.ap50[pos]), 6),
            "map50_95": round(float(box.ap[pos]), 6),
        }

    mean = box.mean_results()
    overall = {
        "precision": round(float(mean[0]), 6),
        "recall": round(float(mean[1]), 6),
        "map50": round(float(mean[2]), 6),
        "map50_95": round(float(mean[3]), 6),
    }
    overall["f1"] = round(
        0.0 if overall["precision"] + overall["recall"] == 0
        else 2 * overall["precision"] * overall["recall"]
        / (overall["precision"] + overall["recall"]), 6
    )

    for row in list(per_class.values()) + [overall]:
        for key, value in row.items():
            if key == "class_id":
                continue
            assert math.isfinite(value), f"non-finite {key}: {value}"

    return per_class, overall


def speed_report(results):
    speed = dict(results.speed)
    total_ms = sum(float(speed.get(k, 0.0)) for k in ("preprocess", "inference", "postprocess"))
    return {
        "preprocess_ms": round(float(speed.get("preprocess", 0.0)), 3),
        "inference_ms": round(float(speed.get("inference", 0.0)), 3),
        "postprocess_ms": round(float(speed.get("postprocess", 0.0)), 3),
        "total_ms_per_image": round(total_ms, 3),
        "fps": round(1000.0 / total_ms, 3) if total_ms > 0 else None,
    }


def print_table(per_class, overall):
    print(f"{'class':<14}{'P':>9}{'R':>9}{'F1':>9}{'mAP50':>9}{'mAP50-95':>11}")
    for name, v in per_class.items():
        print(f"{name:<14}{v['precision']:>9.4f}{v['recall']:>9.4f}{v['f1']:>9.4f}"
              f"{v['map50']:>9.4f}{v['map50_95']:>11.4f}")
    print(f"{'ALL':<14}{overall['precision']:>9.4f}{overall['recall']:>9.4f}"
          f"{overall['f1']:>9.4f}{overall['map50']:>9.4f}{overall['map50_95']:>11.4f}")


val_per_class, val_overall = extract_metrics(val_results, v2_names)
test_per_class, test_overall = extract_metrics(test_results, v2_names)
test_speed = speed_report(test_results)

print("=== validation split ({n} images) ===".format(n=counts["val"]["images"]))
print_table(val_per_class, val_overall)

print("\n=== target test split ({n} images, 6 of them webcam stills) ===".format(n=counts["test"]["images"]))
print_table(test_per_class, test_overall)
print("\nspeed:", test_speed)

## 15. Write the two metrics JSON reports

Two files, both in the V2 reporting schema. This schema is **separate** from the
frozen `experiments/ai_metrics.csv` schema (AI-01): the V2 result must never be
appended there as an AI-01 row.

Missing values are written as `null`, never invented as `0.0`.

- `/content/domain_v2_val_report.json` - validation split.
- `/content/domain_v2_test_report.json` - first and only target test result.

In [ ]:
from datetime import datetime, timezone

def confusion_of(results):
    try:
        return [[int(c) for c in row] for row in results.confusion_matrix.matrix]
    except Exception as exc:
        print("confusion matrix unavailable:", exc)
        return None


def build_report(split, per_class, overall, speed, results, images, caveat):
    return {
        "report": "domain_v2_finetuning",
        "recorded_at": datetime.now(timezone.utc).isoformat(),
        "model": "yolov8n-domain-v2",
        "task": "detect",
        "start_weights": {
            "file": "best.pt",
            "role": "V1 baseline checkpoint (transfer-learning start)",
            "sha256": v1_sha256,
        },
        "weights": str(BEST_V2),
        "dataset": "eye_shield_domain_v2",
        "dataset_root": str(DATASET_ROOT),
        "split": split,
        "split_counts": counts,
        "class_boxes": class_boxes,
        "split_images": images,
        "webcam_images_in_split": WEBCAM_IN_SPLIT.get(split),
        "classes": {str(k): v for k, v in v2_names.items()},
        "train_args": {k: v for k, v in TRAIN_ARGS.items()},
        "train_wall_minutes": round(wall_minutes, 2),
        "eval": {"imgsz": IMGSZ, "batch": BATCH, "device": "cuda:0"},
        "metrics": {
            "per_class": per_class,
            "overall": overall,
            "speed": speed,
        },
        "confusion_matrix": confusion_of(results),
        "environment": env,
        "caveat": caveat,
    }


WEBCAM_IN_SPLIT = {
    "train": 42,
    "val": 12,
    "test": 6,
}

VAL_REPORT = Path("/content/domain_v2_val_report.json")
TEST_REPORT = Path("/content/domain_v2_test_report.json")

val_doc = build_report(
    "val", val_per_class, val_overall, speed_report(val_results), val_results,
    counts["val"]["images"],
    "Validation split of Domain V2. This split was visible to early stopping, "
    "so these numbers are diagnostics, not a field estimate.",
)
TEST_REPORT_DOC = build_report(
    "test", test_per_class, test_overall, test_speed, test_results,
    counts["test"]["images"],
    "Measured once on the 46-image Domain V2 test split, of which only 6 are "
    "webcam stills (3 goggles, 3 no_goggles). The test split was not used for "
    "training, early stopping, threshold selection, augmentation selection or "
    "hyper-parameter tuning. Six webcam images cannot certify field performance; "
    "treat this as regression evidence and validate with the live protocol.",
)

VAL_REPORT.write_text(json.dumps(val_doc, indent=2))
TEST_REPORT.write_text(json.dumps(TEST_REPORT_DOC, indent=2))

print(VAL_REPORT, f"({VAL_REPORT.stat().st_size:,} bytes)")
print(TEST_REPORT, f"({TEST_REPORT.stat().st_size:,} bytes)")
print("\noverall (test):", json.dumps(test_overall, indent=2))

## 16. Sample predictions on webcam stills

Six webcam test images is a small sample, so eyeball them. Each prediction card
shows the boxes the fine-tuned model draws; a `no_goggles` face that is drawn as
`goggles` is the exact false-safe failure this run is trying to fix.

Output goes to `/content/runs/eye_shield_domain_v2_predictions/`.

In [ ]:
import cv2
import numpy as np

PRED_DIR = Path("/content/runs/eye_shield_domain_v2_predictions")
if PRED_DIR.exists():
    shutil.rmtree(PRED_DIR)
PRED_DIR.mkdir(parents=True)

webcam_test = sorted(
    p for p in images_in("test") if "webcam" in p.name.lower()
)
print(f"webcam stills in the test split: {len(webcam_test)}")
for p in webcam_test:
    print("  ", p.name)
assert webcam_test, "no webcam stills found in the test split - check the dataset"

preds = v2_model.predict(
    source=[str(p) for p in webcam_test],
    imgsz=IMGSZ,
    conf=0.25,
    iou=0.45,
    device=0,
    save=True,
    project=str(PRED_DIR.parent),
    name=PRED_DIR.name,
    exist_ok=True,
    verbose=False,
)

for path, res in zip(webcam_test, preds):
    labels = []
    for box in res.boxes:
        cid = int(box.cls.item())
        labels.append(f"{v2_names[cid]} {float(box.conf.item()):.2f}")
    print(f"{path.name}: {labels if labels else 'no detections'}")

# Contact sheet so all six stills can be checked in one look.
thumbs = []
for res in preds:
    img = res.plot()
    h, w = img.shape[:2]
    scale = 320 / max(h, w)
    thumbs.append(cv2.resize(img, (int(w * scale), int(h * scale))))

if thumbs:
    sheet_h = max(t.shape[0] for t in thumbs)
    sheet_w = sum(t.shape[1] for t in thumbs)
    sheet = np.full((sheet_h, sheet_w, 3), 255, dtype=np.uint8)
    x = 0
    for t in thumbs:
        sheet[: t.shape[0], x : x + t.shape[1]] = t
        x += t.shape[1]
    CONTACT_SHEET = PRED_DIR / "webcam_contact_sheet.jpg"
    cv2.imwrite(str(CONTACT_SHEET), sheet)
    print("\ncontact sheet:", CONTACT_SHEET, f"({CONTACT_SHEET.stat().st_size/1024:.0f} KiB)")
else:
    CONTACT_SHEET = None
    print("\nno predictions to compose")

## 17. Package every artifact for download

`/content/eye_shield_v2_results.zip` contains only files that actually exist -
missing artifacts are reported, never stubbed. Contents and size are printed so
the bundle can be checked without unzipping.

In [ ]:
BUNDLE = Path("/content/eye_shield_v2_results")
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
BUNDLE.mkdir(parents=True)

(BUNDLE / "weights").mkdir()
shutil.copy2(BEST_V2, BUNDLE / "weights" / "best_v2.pt")
shutil.copy2(RUN_DIR / "weights" / "last.pt", BUNDLE / "weights" / "last_v2.pt")
shutil.copy2(VAL_REPORT, BUNDLE / VAL_REPORT.name)
shutil.copy2(TEST_REPORT, BUNDLE / TEST_REPORT.name)
(BUNDLE / "environment.json").write_text(json.dumps(env, indent=2))
(BUNDLE / "train_args.json").write_text(json.dumps(TRAIN_ARGS, indent=2))

missing = []

def collect(src_dir, dest_rel, patterns):
    if not src_dir.is_dir():
        missing.append(f"{dest_rel} (run directory absent: {src_dir})")
        return
    dest = BUNDLE / dest_rel
    dest.mkdir(parents=True, exist_ok=True)
    hits = set()
    for pattern in patterns:
        for p in src_dir.glob(pattern):
            if p.is_file():
                shutil.copy2(p, dest / p.name)
                hits.add(p.name)
    if not hits:
        missing.append(dest_rel)

collect(RUN_DIR, "train", ["results.png", "results.csv", "confusion_matrix.png",
                           "confusion_matrix_normalized.png", "*F1_curve.png",
                           "*PR_curve.png", "*P_curve.png", "*R_curve.png",
                           "labels.jpg", "args.yaml", "arg.yaml"])
collect(Path("/content/runs/domain_v2_val_eval"), "val_eval",
        ["confusion_matrix.png", "confusion_matrix_normalized.png",
         "*F1_curve.png", "*PR_curve.png", "*P_curve.png", "*R_curve.png"])
collect(Path("/content/runs/domain_v2_test_eval"), "test_eval",
        ["confusion_matrix.png", "confusion_matrix_normalized.png",
         "*F1_curve.png", "*PR_curve.png", "*P_curve.png", "*R_curve.png"])
collect(PRED_DIR, "predictions", ["*.jpg", "*.png"])

if missing:
    print("MISSING artifacts (not fabricated):")
    for m in missing:
        print("  -", m)

files_list = sorted(p for p in BUNDLE.rglob("*") if p.is_file())
print(f"\nbundle: {len(files_list)} files")
for p in files_list:
    print(f"  {p.relative_to(BUNDLE)}  ({p.stat().st_size/1024:.0f} KiB)")

RESULTS_ZIP = shutil.make_archive("/content/eye_shield_v2_results", "zip", BUNDLE)
print(f"\n{RESULTS_ZIP}  ({Path(RESULTS_ZIP).stat().st_size/1024/1024:.2f} MB)")

## 18. Download and install locally

Download `eye_shield_v2_results.zip`, then follow the five steps below on the
Mac. **Nothing in this list overwrites `models/best.pt`, the AI-01 metrics row,
the locked test dataset, or the baseline notebook.**

In [ ]:
from google.colab import files as colab_files

colab_files.download(RESULTS_ZIP)

print(
    """
Local install, five steps (run on the Mac, from the repo root):

  1. Unzip:            unzip -o ~/Downloads/eye_shield_v2_results.zip -d /tmp/v2
  2. Check weights:    ls -l /tmp/v2/weights/best_v2.pt
  3. Promote (only after the live check passes):
       cp /tmp/v2/weights/best_v2.pt models/best_v2.pt
     (V1 stays at models/best.pt. Never overwrite it with V2.)
  4. Point the demo at V2 without touching V1:
       WEIGHTS_PATH=models/best_v2.pt .venv/bin/python tools/live_eye_shield_demo.py
  5. Live acceptance before any V2 claim: 10/5/5 goggles (front/left/right),
     10/5/5 no_goggles (front/left/right) and 5 empty scenes. Reject on the
     first false-safe (bare eyes reported as goggles).
"""
)

## 19. Checklist before you report any V2 number

- [ ] `environment.json` records the GPU actually used (T4 expected).
- [ ] `domain_v2_test_report.json` exists - it is the **only** source of V2
      numbers. Nothing is quoted from memory or from the training log.
- [ ] The test split was evaluated **once**. If you re-trained after looking at
      the test metrics, the split is contaminated and the report must say so.
- [ ] The report's `caveat` field is quoted alongside the numbers: 46 test
      images, of which 6 are webcam stills.
- [ ] The confusion matrix is inspected - specifically the `no_goggles → goggles`
      cell, because that is the false-safe direction.
- [ ] V1's AI-01 row in `experiments/ai_metrics.csv` is unchanged. The V2 result
      is **not** appended there.
- [ ] `models/best.pt` on the Mac is still V1 (SHA-256 `9366b73b…fa3591`).
- [ ] Live acceptance has run before V2 is described as shipped.
- [ ] Targets are cited as targets: P ≥ 0.90, R ≥ 0.85, F1 ≥ 0.87,
      mAP@0.5 ≥ 0.90, ≥ 15 FPS.